In [9]:
import pandas as pd
from pathlib import Path

# Extract SEF headers to Excel

In [10]:
root = Path("/scratch3/PALAEO-RA/daily_data/final/")
rows = []

files = list(root.rglob("*.tsv")) # gets all tsv files recursively

def clean_line(line):
    line = line.rstrip("\n")           # remove newline
    # Split on tabs but keep empty fields
    parts = line.split("\t")
    # Remove empty fields at the end only
    while parts and parts[-1] == "":
        parts.pop()
    # Rejoin back with single tabs
    return "\t".join(parts)


for path in files:
    
    # ------------ read metadata until "Year" ------------
    meta = {}
    
    # read whole file
    with path.open("r", encoding="latin1") as f:
        for raw in f:
            line = clean_line(raw)
            # stop at start of the actual SEF table
            if line.startswith("Year"):
                break
            if "\t" in line:
                key, val = line.split("\t", 1)
                meta[key.strip()] = val.strip()
    
    # ------------ detect table header row ------------
    with path.open("r", encoding="latin1") as f:
        table_start = None
        for i, raw in enumerate(f):
            line = clean_line(raw)
            if line.startswith("Year"):
                table_start = i
                break

    # ------------ read the table safely ------------
    df_file = pd.read_csv(
        path,
        sep="\t",
        header=table_start,
        encoding="latin-1",
        dtype=str,
        on_bad_lines="skip"
    )

    # ------------ compute start/end date ------------
    meta["filename"]   = path.name
    meta["location"]   = path.parent.name
    meta["start_date"] = df_file["Year"].iloc[0] + "-" + df_file["Month"].iloc[0] + "-" + df_file["Day"].iloc[0]
    meta["end_date"]   = df_file["Year"].iloc[-1] + "-" + df_file["Month"].iloc[-1] + "-" + df_file["Day"].iloc[-1]

    rows.append(meta)


Detect problematic files

In [11]:
# Define the set of 'standard' expected columns for your SEF data
expected_cols = {'Year', 'Month', 'Day', 'Hour', 'Minute', 'Period', 'Value', 'Meta'}

problematic_files = []

for path in files:

    try:
        # ... [Keep your code to find table_start] ...

        # Read the file
        df_file = pd.read_csv(
            path,
            sep="\t",
            header=table_start,
            encoding="latin-1",
            dtype=str,
            on_bad_lines="warn" # Change to warn so you see errors in console
        )

        # CHECK: Does this file have columns that shouldn't be there?
        actual_cols = set(df_file.columns)
        extra_cols = actual_cols - expected_cols
        
        if extra_cols:
            print(f"!!! Problem detected in: {path.name}")
            print(f"    Extra columns found: {extra_cols}")
            problematic_files.append({
                "path": str(path),
                "extra_columns": list(extra_cols)
            })
            continue # Skip adding this one to 'rows' so it doesn't break your final df

    except Exception as e:
        print(f"Error processing {path.name}: {e}")

# Summary of the "Mechanicals"
print(f"\nTotal problematic files found: {len(problematic_files)}")

!!! Problem detected in: UA-UHMI_Odesa_18400101-18500613_p_subdaily.tsv
    Extra columns found: {'Unnamed: 10', 'Unnamed: 9', 'Unnamed: 13', 'Unnamed: 12', 'Unnamed: 8', 'Unnamed: 11'}
!!! Problem detected in: UA-UHMI_Luhansk_18371221-18390112_p_subdaily.tsv
    Extra columns found: {'Unnamed: 10', 'Unnamed: 9', 'Unnamed: 13', 'Unnamed: 12', 'Unnamed: 8', 'Unnamed: 11'}

Total problematic files found: 2


check the problematic columns and the files where they come from

In [12]:
df = pd.DataFrame(rows)

# drop any duplicated rows based on filename (this weird 1857 duplicates from Sorve)
df_clean = df.drop_duplicates(subset=["filename", "location"]).reset_index(drop=True)

# sort by location alphabetically
df_clean = df_clean.sort_values(by="location").reset_index(drop=True)

# move location to the first column
cols = ["location"] + [col for col in df_clean.columns if col != "location"]
df_clean = df_clean[cols]
print(df_clean.columns)

# List of the 'stupid' columns found in your index
extra_cols = ['SEF | obs.time=8a.m.', '| QC software=dataresqc v1.1.1SEF', '15.SEF']

# Check which files have non-null values in these columns
for col in extra_cols:
    if col in df.columns:
        files_with_data = df[df[col].notna()]['filename'].unique()
        print(f"Column '{col}' contains data from: {files_with_data}")

df_clean.to_csv("/scratch3/PALAEO-RA/daily_data/final/metadata_summary.csv", index=False)
df_clean.to_csv("/scratch2/ccorbella/code/dataprep/metadata_summary.csv", index=False)

df_clean.head()
df_clean.tail()

Index(['location', 'SEF', 'ID', 'Name', 'Lat', 'Lon', 'Alt', 'Source', 'Link',
       'Vbl', 'Stat', 'Units', 'filename', 'start_date', 'end_date', 'Meta'],
      dtype='object')


,location,SEF,ID,Name,Lat,Lon,Alt,Source,Link,Vbl,Stat,Units,filename,start_date,end_date,Meta
1786,Zwanenburg,1.0.0,1,Zwanenburg,52.383,4.75,1,DeBilt,NaN,p,point,hPa,DeBilt_Zwanenburg_DeBilt_17430101-18501231_p_s...,1743-1-1,1850-12-31,QC software=dataresqc v1.1.1
1787,Zwanenburg,1.0.0,KNMI-Zwanenburg,Zwanenburg,52.32,4.74,1,KNMI,https://www.knmi.nl/nederland-nu/klimatologie/...,rr,point,mm,KNMI_Zwanenburg_17380114-18601130_rr_subdaily_...,1738-1-14,1860-11-30,QC software=dataresqc v1.1.1
1788,Zwanenburg,1.0.0,KNMI-Zwanenburg,Zwanenburg,52.32,4.74,1,KNMI,https://www.knmi.nl/nederland-nu/klimatologie/...,dd,point,degree,KNMI_Zwanenburg_17380101-18601231_dd_subdaily_...,1738-1-1,1860-12-31,QC software=dataresqc v1.1.1
1789,Zwanenburg,1.0.0,KNMI-Zwanenburg,Zwanenburg,52.32,4.74,1,KNMI,https://www.knmi.nl/nederland-nu/klimatologie/...,p,point,hPa,KNMI_Zwanenburg_17380101-18601231_p_subdaily_q...,1738-1-1,1860-12-31,PTC=Y | PGC=Y | QC software=dataresqc v1.1.1
1790,Zwanenburg,1.0.0,KNMI-Zwanenburg,Zwanenburg,52.32,4.74,1,KNMI,https://www.knmi.nl/nederland-nu/klimatologie/...,ta,point,C,KNMI_Zwanenburg_17380101-18601231_ta_subdaily_...,1738-1-1,1860-12-31,QC software=dataresqc v1.1.1


check if there are duplicate columns

sometimes restarting the kernel gets rid of them, as they are stored in meta()

In [13]:
duplicates = df[df.duplicated(subset=["filename", "location"], keep=False)]

# Sort by filename so the copies are grouped together visually
duplicates = duplicates.sort_values(by=["filename", "location"])

print(f"Total duplicated rows found: {len(duplicates)}")
display(duplicates)

Total duplicated rows found: 0


,SEF,ID,Name,Lat,Lon,Alt,Source,Link,Vbl,Stat,Units,filename,location,start_date,end_date,Meta


**find adjacent files that should be counted as 1**

In [14]:
df2 = df.copy()

# round coords to 3 decimals
df2["Lat3"] = df2["Lat"].round(3)
df2["Lon3"] = df2["Lon"].round(3)

# sort so adjacency makes sense
df2 = df2.sort_values(["Vbl", "Stat", "Units", "Lat3", "Lon3", "start_date"])
df2["start_int"] = df2["start_date"].str.replace("-", "").astype(int)
df2["end_int"]   = df2["end_date"].str.replace("-", "").astype(int)

# group key
grp_cols = ["Vbl", "Stat", "Units", "Lat3", "Lon3"]

# check adjacency
for key, sub in df2.groupby(grp_cols):
    sub = sub.sort_values("start_date")

    for i in range(len(sub)-1):
        end_i = sub.iloc[i]["end_date"]
        start_j = sub.iloc[i+1]["start_date"]

        # approximate check: same year or next year AND day-of-year difference < 366
        sy = int(str(start_j)[:4])
        ey = int(str(end_i)[:4])

        if (sy == ey or sy == ey+1):
            f1 = sub.iloc[i]["filename"]
            f2 = sub.iloc[i+1]["filename"]
            print(f"WARNING: {f1} and {f2} can be grouped (end={end_i}, start={start_j})")

The code to group adjacent files is `sef_series_length_v2.py`. 

# Table where is the oldest Record

In [15]:
df = pd.read_csv("/scratch3/PALAEO-RA/daily_data/final/metadata_summary.csv")

In [16]:
date_parts = df['start_date'].str.split('-', expand=True).astype(int)
df['start_year'] = date_parts[0]
df['start_month'] = date_parts[1]
df['start_day'] = date_parts[2]

# numeric key to compare without needing datetime
df['start_ymd'] = df['start_year'] * 10000 + df['start_month'] * 100 + df['start_day']

# find index of row with oldest day per var
idx = df.groupby('Vbl')['start_ymd'].idxmin()

oldest_records = df.loc[idx, ['Vbl', 'Name', 'start_date', 'Source']].copy()
oldest_records = oldest_records.sort_values(by='Vbl').reset_index(drop=True)

# only show some of the variables of choice and in this order
vars_of_interest = ['ta', 'p', 'dd', 'Tx', 'Tn', 'rr']

sel = oldest_records[oldest_records['Vbl'].isin(vars_of_interest)]
sel = sel.set_index('Vbl').loc[vars_of_interest].reset_index()
# sel['Vbl'] =  sel['Vbl'].map(mapping)

sel

latex_table = sel.to_latex(
    index=False,
    column_format="llll"  # adjust if you have more columns
)

print(latex_table)


\begin{tabular}{llll}
\toprule
Vbl & Name & start_date & Source \\
\midrule
ta & Paris_Rousseau & 1658-5-25 & Rousseau \\
p & Oxford & 1666-7-4 & PALAEO-RA \\
dd & Oxford & 1666-7-4 & PALAEO-RA \\
Tx & Paris_Rousseau & 1675-12-1 & Rousseau \\
Tn & Paris_Rousseau & 1675-12-1 & Rousseau \\
rr & Paris & 1665-2-1 & Pliemon, T., Foelsche, U., Rohr, C., & Pfister, C. (2023). Precipitation reconstructions for Paris based on the observations by Louis Morin, 1665â1713 CE [Data set]. \\
\bottomrule
\end{tabular}

